# Parallelization
# Parallelization with LLMs

With parallelization, LLMs work simultaneously on a task.  
This is either done by:

- Running multiple independent **subtasks** at the same time  
- Running the same task multiple times to check for **different outputs**

---

## Benefits of Parallelization
- **Split subtasks** and run them in parallel → increases **speed**  
- **Run tasks multiple times** to compare outputs → increases **confidence**  

---

## Examples
1. Running one subtask that processes a document for **keywords**, and a second subtask to check for **formatting errors**  
2. Running a task multiple times that scores a document for **accuracy** based on:  
   - Number of citations  
   - Number of sources used  
   - Quality of the sources  



In [4]:
from utils import get_groq_llm
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Dict, Any

In [5]:
llm = get_groq_llm()

In [18]:
class QueryAnalysisState(TypedDict):
    user_query: str
    title: str | None
    description: str | None
    linkedinpost: str | None
    hashtags: str| None
    final_analysis: dict| None

In [19]:
title_generator_prompt = ChatPromptTemplate.from_template(
    "Write me one good Youtube Video Title for this {query}. Return with only title and no other Text"
)
description_prompt = ChatPromptTemplate.from_template(
    "Write me One Youtube Video Description for this {query}.Return with only description and no other Text"
)
linkedin_post_prompt = ChatPromptTemplate.from_template(
    "I made a Youtube Video on {query}. Now write me a LinkedIn post regarding the topic, give some knowledge and facts about the topic then route it to YT Video. Return with only Post text and no other Text"
)
hashtag_generator = ChatPromptTemplate.from_template(
    "Generate me relevant hashtags for video related to {query} on Youtube. Provide me 30 hashtags. Return with only hashtags and no other Text"
)

In [20]:
def generate_title(state: QueryAnalysisState):
    response = (title_generator_prompt | llm).invoke({"query": state["user_query"]})
    return {"title": response.content}

def generate_description(state: QueryAnalysisState):
    response = (description_prompt | llm).invoke({"query": state["user_query"]})
    return {"description": response.content}

def linkedin_post(state: QueryAnalysisState):
    response = (linkedin_post_prompt | llm).invoke({"query": state["user_query"]})
    return {"linkedinpost": response.content}

def generate_hashtags(state: QueryAnalysisState):
    response = (hashtag_generator | llm).invoke({"query": state["user_query"]})
    return {"hashtags": response.content}
    
def merge_results(state: QueryAnalysisState):
    return {
        "final_analysis": {
            "title": state["title"],
            "description": state["description"],
            "linkedinpost": state["linkedinpost"],
            "hashtags": state["hashtags"],
        }
    }

In [21]:
builder = StateGraph(QueryAnalysisState)

builder.add_node("generate_title", generate_title)
builder.add_node("generate_description", generate_description)
builder.add_node("linkedin_post", linkedin_post)
builder.add_node("generate_hashtags", generate_hashtags)
builder.add_node("merge", merge_results)

In [22]:
builder.add_edge("__start__", "generate_title")
builder.add_edge("__start__", "generate_description")
builder.add_edge("__start__", "linkedin_post")
builder.add_edge("__start__", "generate_hashtags")

builder.add_edge("generate_title", "merge")
builder.add_edge("generate_description", "merge")
builder.add_edge("linkedin_post", "merge")
builder.add_edge("generate_hashtags", "merge")
builder.add_edge("merge", END)

graph = builder.compile(checkpointer=MemorySaver())

In [23]:

result = graph.invoke({"user_query": "A video on Langgraph with FastAPI Backend"},
                     config={"configurable": {"thread_id": "session-1"}})
print(f"Title: {result["final_analysis"]['title']} \n\nDescription: {result["final_analysis"]['description']} \n\nLinkedIn Post: {result["final_analysis"]['linkedinpost']} \n\nHashtags: {result["final_analysis"]['hashtags']}")

Title: "Building a Real-Time LangGraph API with FastAPI: A Scalable Natural Language Processing Backend" 

Description: "Build a High-Performance API with Langraph and FastAPI

In this tutorial, we'll be exploring the world of Langraph, a powerful graph database, and connecting it to a FastAPI backend. Learn how to model and query complex graph structures, and build a scalable API using Python and the FastAPI framework.

We'll cover:

- Setting up a Langraph database and connecting it to a FastAPI app
- Modeling complex graph structures using Langraph's Python SDK
- Querying and traversing graph data with Langraph's query language
- Building a RESTful API using FastAPI to interact with the Langraph database
- Deploying the API to a production-ready environment

Whether you're a graph database newbie or an experienced developer looking to scale your API, this tutorial has got you covered. So, let's get started and build a high-performance API with Langraph and FastAPI!" 

LinkedIn Post: